# Import Libraries

In [29]:
import pandas as pd
from functools import reduce
import os

In [30]:
files = {
    "Core_CPI": "Core CPI, seas. adj..xlsx",
    
    "Inflation":
    "CPI Price, % y-o-y, median weighted, seas. adj..xlsx",
    
    "Exports":
    "Exports Merchandise, Customs, constant 2010 US$, millions, seas. adj..xlsx",
    
    "Foreign_Reserves":
    "Foreign Reserves, Months Import Cover, Goods.xlsx",
    
    "GDP":
    "GDP at market prices, constant 2010 US$, millions, seas. adj..xlsx",
    
    "Imports":
    "Imports Merchandise, Customs, constant 2010 US$, millions, seas. adj..xlsx",
    
    "Industrial_Production":
    "Industrial Production, constant 2010 US$, seas. adj..xlsx",
    
    "Exchange_Rate":
    "Official exchange rate, LCU per USD, period average.xlsx",
    
    "Retail_Sales":
    "Retail Sales Volume Index, seas. adj..xlsx",
    
    "Stock_Market":
    "Stock Markets, US$.xlsx",
    
    "Terms_of_Trade":
    "Terms of Trade.xlsx",
    
    "Total_Reserves":
    "Total Reserves.xlsx",
    
    "Unemployment":
    "Unemployment Rate, seas. adj..xlsx"
}

In [31]:
#function
def process_file(filepath, value_name):
    
    df = pd.read_excel(filepath)

    df = df.rename(columns={df.columns[0]: "Year"})
    df = df.dropna(how="all")

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year"])
    df["Year"] = df["Year"].astype(int)

    df_long = df.melt(
        id_vars="Year",
        var_name="Country",
        value_name=value_name
    )

    df_long["Country"] = df_long["Country"].astype(str).str.strip()

    return df_long

In [32]:
#Process All Files
all_dfs = []

for variable_name, filepath in files.items():
    
    print(f"Processing: {variable_name}")
    
    df_processed = process_file(filepath, variable_name)
    
    all_dfs.append(df_processed)

Processing: Core_CPI
Processing: Inflation
Processing: Exports
Processing: Foreign_Reserves
Processing: GDP
Processing: Imports
Processing: Industrial_Production
Processing: Exchange_Rate
Processing: Retail_Sales
Processing: Stock_Market
Processing: Terms_of_Trade
Processing: Total_Reserves
Processing: Unemployment


In [35]:
#MERGE ALL DATASETS
merged_df = reduce(
    lambda left, right:
    pd.merge(
        left,
        right,
        on=["Year", "Country"],
        how="outer"
    ),
    all_dfs
)

In [36]:
#Cleaning
# Remove duplicated rows
merged_df = merged_df.drop_duplicates()

# Remove rows without country/year
merged_df = merged_df.dropna(
    subset=["Year", "Country"]
)

# Reset index
merged_df = merged_df.reset_index(drop=True)

In [37]:
merged_df.head()

,Year,Country,Core_CPI,Inflation,Exports,Foreign_Reserves,GDP,Imports,Industrial_Production,Exchange_Rate,Retail_Sales,Stock_Market,Terms_of_Trade,Total_Reserves,Unemployment
0,1996,Advanced Economies,NaN,NaN,4520414.0,NaN,NaN,4989094.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1996,Algeria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1996,Argentina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1996,Australia,NaN,NaN,114075.7,NaN,NaN,74357.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1996,Austria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
#Save dataset
merged_df.to_csv(
    "global_economy_dataset.csv",
    index=False
)

print("\nDataset successfully merged!")
print(merged_df.head())
print("\nShape:", merged_df.shape)


Dataset successfully merged!
   Year             Country  Core_CPI  Inflation    Exports  Foreign_Reserves  \
0  1996  Advanced Economies       NaN        NaN  4520414.0               NaN   
1  1996             Algeria       NaN        NaN        NaN               NaN   
2  1996           Argentina       NaN        NaN        NaN               NaN   
3  1996           Australia       NaN        NaN   114075.7               NaN   
4  1996             Austria       NaN        NaN        NaN               NaN   

   GDP     Imports  Industrial_Production  Exchange_Rate  Retail_Sales  \
0  NaN  4989094.00                    NaN            NaN           NaN   
1  NaN         NaN                    NaN            NaN           NaN   
2  NaN         NaN                    NaN            NaN           NaN   
3  NaN    74357.23                    NaN            NaN           NaN   
4  NaN         NaN                    NaN            NaN           NaN   

   Stock_Market  Terms_of_Trade  Total

# Missing Value Impute

In [55]:

df=pd.read_csv("global_economy_dataset.csv")

In [56]:
df.shape

(6449, 15)

In [57]:
#drop terms_of_trade
df = df.drop(columns=["Terms_of_Trade"])

In [58]:
# Drop Terms_of_Trade karena kosong total / terlalu banyak missing
df = df.drop(columns=["Terms_of_Trade"], errors="ignore")

# Sort dulu
df = df.sort_values(["Country", "Year"])

# Forward fill + backward fill per country
df = (
    df.groupby("Country", group_keys=False)
      .apply(lambda x: x.ffill().bfill())
)

# Reset index
df = df.reset_index(drop=True)

# Cek missing lagi
df.isnull().sum()

C:\Users\62853\AppData\Local\Temp\ipykernel_11912\2503929889.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.ffill().bfill())


Year                        0
Country                     0
Core_CPI                 4491
Inflation                6077
Exports                  4868
Foreign_Reserves         2851
GDP                      3454
Imports                  4775
Industrial_Production    3666
Exchange_Rate             432
Retail_Sales             5339
Stock_Market             4062
Total_Reserves            901
Unemployment             4088
dtype: int64

In [59]:
# Keep row yang minimal punya 70% data lengkap
threshold = int(df.shape[1] * 0.7)

df_clean = df.dropna(thresh=threshold)

print(df_clean.isnull().sum())
print(df_clean.shape)

Year                        0
Country                     0
Core_CPI                  678
Inflation                1965
Exports                   756
Foreign_Reserves           60
GDP                        62
Imports                   663
Industrial_Production     154
Exchange_Rate             310
Retail_Sales             1165
Stock_Market              339
Total_Reserves              0
Unemployment              216
dtype: int64
(2275, 14)


In [60]:
#Impute using median
numeric_cols = df_clean.select_dtypes(include="number").columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

C:\Users\62853\AppData\Local\Temp\ipykernel_11912\325003991.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(
C:\Users\62853\AppData\Local\Temp\ipykernel_11912\325003991.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(
C:\Users\62853\AppData\Local\Temp\ipykernel_11912\325003991.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

In [61]:
df_clean.isnull().sum()

Year                     0
Country                  0
Core_CPI                 0
Inflation                0
Exports                  0
Foreign_Reserves         0
GDP                      0
Imports                  0
Industrial_Production    0
Exchange_Rate            0
Retail_Sales             0
Stock_Market             0
Total_Reserves           0
Unemployment             0
dtype: int64

In [62]:
df_clean.head()

,Year,Country,Core_CPI,Inflation,Exports,Foreign_Reserves,GDP,Imports,Industrial_Production,Exchange_Rate,Retail_Sales,Stock_Market,Total_Reserves,Unemployment
0,1996,Advanced Economies,101.8832,2.113055,4520414.0,0.291661,33950163.0,4989094.0,8.100000e+12,7.546392,59.23351,93.097505,1202061.0,7.071621
1,1997,Advanced Economies,101.8832,2.113055,4952759.0,0.291661,33950163.0,5392757.0,8.100000e+12,7.546392,59.23351,93.097505,1202061.0,7.071621
2,1998,Advanced Economies,101.8832,1.909170,5141825.0,0.298636,34821159.0,5658476.0,8.250000e+12,7.546392,57.30905,93.097505,1230332.0,6.683189
3,1999,Advanced Economies,101.8832,1.737984,5409369.0,0.290025,35985399.0,6105051.0,8.500000e+12,7.546392,60.13630,93.097505,1261871.0,6.379832
4,2000,Advanced Economies,101.8832,2.701247,6064090.0,0.280050,37467104.0,6773346.0,8.920000e+12,7.546392,59.98409,93.097505,1366128.0,5.945781


In [64]:
#save new dataset
df_clean.to_csv(
    "global_economy_clean_dataset.csv",
    index=False
)

print("\nDataset successfully merged!")
print(df_clean.head())
print("\nShape:", df_clean.shape)


Dataset successfully merged!
   Year             Country  Core_CPI  Inflation    Exports  Foreign_Reserves  \
0  1996  Advanced Economies  101.8832   2.113055  4520414.0          0.291661   
1  1997  Advanced Economies  101.8832   2.113055  4952759.0          0.291661   
2  1998  Advanced Economies  101.8832   1.909170  5141825.0          0.298636   
3  1999  Advanced Economies  101.8832   1.737984  5409369.0          0.290025   
4  2000  Advanced Economies  101.8832   2.701247  6064090.0          0.280050   

          GDP    Imports  Industrial_Production  Exchange_Rate  Retail_Sales  \
0  33950163.0  4989094.0           8.100000e+12       7.546392      59.23351   
1  33950163.0  5392757.0           8.100000e+12       7.546392      59.23351   
2  34821159.0  5658476.0           8.250000e+12       7.546392      57.30905   
3  35985399.0  6105051.0           8.500000e+12       7.546392      60.13630   
4  37467104.0  6773346.0           8.920000e+12       7.546392      59.98409   

  

# EDA PROCESS

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

In [65]:
df = pd.read_csv("global_economy_clean_dataset.csv")
df.head()

,Year,Country,Core_CPI,Inflation,Exports,Foreign_Reserves,GDP,Imports,Industrial_Production,Exchange_Rate,Retail_Sales,Stock_Market,Total_Reserves,Unemployment
0,1996,Advanced Economies,101.8832,2.113055,4520414.0,0.291661,33950163.0,4989094.0,8.100000e+12,7.546392,59.23351,93.097505,1202061.0,7.071621
1,1997,Advanced Economies,101.8832,2.113055,4952759.0,0.291661,33950163.0,5392757.0,8.100000e+12,7.546392,59.23351,93.097505,1202061.0,7.071621
2,1998,Advanced Economies,101.8832,1.909170,5141825.0,0.298636,34821159.0,5658476.0,8.250000e+12,7.546392,57.30905,93.097505,1230332.0,6.683189
3,1999,Advanced Economies,101.8832,1.737984,5409369.0,0.290025,35985399.0,6105051.0,8.500000e+12,7.546392,60.13630,93.097505,1261871.0,6.379832
4,2000,Advanced Economies,101.8832,2.701247,6064090.0,0.280050,37467104.0,6773346.0,8.920000e+12,7.546392,59.98409,93.097505,1366128.0,5.945781


In [ ]:
#GDP global trend

global_gdp = df.groupby("Year")["GDP"].mean().reset_index()

fig = px.line(
    global_gdp,
    x="Year",
    y="GDP",
    title="Average Global GDP Trend"
)


pio.renderers.default = "browser"

fig.show()

In [ ]:
#Inflation trend

global_inflation = df.groupby("Year")["Inflation"].mean().reset_index()

fig = px.line(
    global_inflation,
    x="Year",
    y="Inflation",
    title="Average Global Inflation Trend"
)

pio.renderers.default = "browser"
fig.show()

In [77]:
#Unemployment trend
global_unemployment = df.groupby("Year")["Unemployment"].mean().reset_index()   
fig = px.line(
    global_unemployment,
    x="Year",
    y="Unemployment",
    title="Average Global Unemployment Trend"
)

pio.renderers.default = "browser"
fig.show()


In [78]:
#country comparison: Top 10 countries by average GDP

top_gdp = df.groupby("Country")["GDP"].mean().nlargest(10).reset_index()    

fig = px.bar(
    top_gdp,
    x="Country",
    y="GDP",
    title="Top 10 Countries by Average GDP"
)

pio.renderers.default = "browser"
fig.show()


In [79]:
#country comparison: Top 10 countries by inflation

top_inflation= df.groupby("Country")["Inflation"].mean().nlargest(10).reset_index()

fig = px.bar(
    top_inflation,
    x="Country",
    y="Inflation",
    title="Top 10 Countries by Average Inflation"
)

pio.renderers.default = "browser"
fig.show()

In [80]:
#Export vs Import trend
trade_global=df.groupby("Year")[["Exports", "Imports"]].mean().reset_index()

fig=px.line(
    trade_global,
    x="Year",
    y=["Exports", "Imports"],
    title="Average Global Exports vs Imports Trend"
)
pio.renderers.default = "browser"
fig.show()

# Feature Engineering

In [81]:
 #Trade Balance
df["Trade_Balance"] = df["Exports"] - df["Imports"]

In [82]:
#Trade Opennes
df["Trade_Openness"] = df["Trade_Balance"] / df["GDP"]

In [83]:
#Reserve Adequacy
df["Reserve_Adequacy"] = df["Total_Reserves"] / df["GDP"]

In [84]:
#Stock Market to GDP Ratio
df["Stock_to_GDP"] = df["Stock_Market"] / df["GDP"]

In [85]:
#GDP Growth
df["GDP_Growth"] = df.groupby("Country")["GDP"].pct_change() * 100

In [86]:
#Export Growth
df["Export_Growth"] = df.groupby("Country")["Exports"].pct_change() * 100

In [87]:
#Import Growth
df["Import_Growth"] = df.groupby("Country")["Imports"].pct_change() * 100

In [88]:
# Stock Market Growth
df["Stock_Growth"] = df.groupby("Country")["Stock_Market"].pct_change() * 100


In [89]:
#Exchange rate change
df["Exchange_Rate_Change"] = df.groupby("Country")["Exchange_Rate"].pct_change() * 100

In [90]:
#Inflation Lag
df["Inflation_Lag"] = df.groupby("Country")["Inflation"].shift(1)

In [91]:
#GDP growth lag
df["GDP_Growth_Lag"] = df.groupby("Country")["GDP_Growth"].shift(1)

In [92]:
#Unemployment Lag
df["Unemployment_Lag"] = df.groupby("Country")["Unemployment"].shift(1)

In [93]:
# 3 year GDP growth average
df["GDP_Growth_Rolling_3Y"] = df.groupby("Country")["GDP_Growth"].rolling(window=3).mean().reset_index(0, drop=True)

In [94]:
#3 year inflation average
df["Inflation_Rolling_3Y"] = df.groupby("Country")["Inflation"].rolling(window=3).mean().reset_index(0, drop=True)

In [96]:
#Clean after feature engineering
import numpy as np
df = df.replace([np.inf, -np.inf], np.nan)

df_model = df.dropna().reset_index(drop=True)

df_model.isnull().sum()
df_model.shape

(2053, 28)

In [97]:
df_model.to_csv("final_global_economy_data.csv", index=False)